# Flight Delay Intelligence - Modeling

Goal: predict whether a flight will arrive 15+ minutes late
(`is_arrival_delayed`).

Sections:
1. Training summary
2. Class distribution & imbalance
3. Metrics comparison
4. Confusion matrix
5. Precision-Recall curve
6. ROC curve
7. Threshold analysis
8. Feature importances
9. Interpretation

In [ ]:
import sys, json
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    confusion_matrix,
    precision_recall_curve,
    roc_curve,
    auc,
)

from config import METRICS_PATH, MODEL_PATH
from features import NUMERIC_FEATURES, CATEGORICAL_FEATURES, TARGET, load_features

sns.set_theme(style='whitegrid')

## 1. Training summary

We trained three pipelines (`logistic_regression`, `random_forest`,
and optionally `xgboost`) using `src/train_model.py`.
Numeric features are standardized (with median imputation for optional weather columns);
categoricals are one-hot encoded.  Each model uses `class_weight='balanced'`
(or `scale_pos_weight` for XGBoost) to counteract the ~80/20 on-time/delayed imbalance.
Best model is selected by ROC AUC; the decision threshold is tuned to target recall >= 40%.

In [ ]:
metrics_payload = json.loads(METRICS_PATH.read_text())
best_name = metrics_payload['best_model']
metrics_df = pd.DataFrame(metrics_payload['metrics']).T.round(4)
print(f'Best model: {best_name}')
metrics_df

## 2. Class distribution & imbalance

The target is heavily imbalanced: roughly 80% of flights arrive on time.
A naive classifier that always predicts 'on-time' would score ~80% accuracy but
**0% recall on actual delays** — useless for travellers.  We address this with
`class_weight='balanced'` and threshold tuning (see section 7).

In [ ]:
df_sample = load_features(sample_size=200_000)
counts = df_sample[TARGET].value_counts().rename({0: 'On Time', 1: 'Delayed'})
pct = counts / counts.sum() * 100

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(counts.index, counts.values, color=['steelblue', 'tomato'])
for bar, p in zip(bars, pct.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 200,
        f'{p:.1f}%',
        ha='center', va='bottom', fontsize=11,
    )
ax.set_title('Class distribution — flight arrivals')
ax.set_ylabel('Flight count')
ax.set_xlabel('')
plt.tight_layout();

## 3. Metrics comparison

In [ ]:
plot_cols = [c for c in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc'] if c in metrics_df.columns]
fig, ax = plt.subplots(figsize=(9, 5))
metrics_df[plot_cols].plot(kind='bar', ax=ax)
ax.set_title('Model comparison (metrics at tuned threshold)')
ax.set_ylabel('Score')
ax.set_ylim(0, 1)
plt.xticks(rotation=0)
plt.legend(loc='lower right')
plt.tight_layout();

## 4. Confusion matrix on a fresh sample

In [ ]:
model = joblib.load(MODEL_PATH)
df = load_features(sample_size=100_000)
X = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y = df[TARGET]
_, X_test, _, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

best_threshold = metrics_payload['metrics'][best_name].get('threshold', 0.5)
y_proba = model.predict_proba(X_test)[:, 1]
y_pred = (y_proba >= best_threshold).astype(int)

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix(y_test, y_pred),
    display_labels=['On Time', 'Delayed'],
).plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title(f'Confusion Matrix — {best_name} (threshold={best_threshold:.2f})')
plt.tight_layout();

## 5. Precision-Recall curve

The PR curve is the right diagnostic for imbalanced classes.  It shows the
precision/recall trade-off across all possible thresholds.  A random classifier
would hug the bottom (PR-AUC ≈ positive-class prevalence).  The vertical dashed
line marks the tuned operating threshold.

In [ ]:
precisions, recalls, thresholds_pr = precision_recall_curve(y_test, y_proba)
pr_auc = auc(recalls, precisions)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(recalls, precisions, color='steelblue', lw=2,
        label=f'{best_name}  (PR-AUC = {pr_auc:.3f})')

# Mark the tuned operating point
op_prec = float(metrics_payload['metrics'][best_name].get('precision', 0))
op_rec  = float(metrics_payload['metrics'][best_name].get('recall', 0))
ax.scatter([op_rec], [op_prec], s=100, zorder=5, color='tomato',
           label=f'Tuned threshold ({best_threshold:.2f})')

# Baseline: random classifier at positive-class prevalence
prevalence = float(y_test.mean())
ax.axhline(prevalence, color='grey', linestyle='--', lw=1,
           label=f'Random classifier (prevalence={prevalence:.2f})')

ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve')
ax.legend(loc='upper right')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
plt.tight_layout();

## 6. ROC curve

ROC-AUC is the primary model selection metric here because it is
threshold-independent.  The diagonal is random-chance performance (AUC = 0.5).

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc_val = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, color='steelblue', lw=2,
        label=f'{best_name}  (AUC = {roc_auc_val:.3f})')
ax.plot([0, 1], [0, 1], color='grey', linestyle='--', lw=1, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve')
ax.legend(loc='lower right')
plt.tight_layout();

## 7. Threshold analysis

Lowering the decision threshold from 0.5 improves recall at the cost of
precision.  The chart below shows how precision, recall, and F1 move as
the threshold varies.  The vertical dashed line is the tuned operating point
(`find_optimal_threshold` in `train_model.py`).

In [ ]:
p_arr = precisions[:-1]
r_arr = recalls[:-1]
f1_arr = 2 * p_arr * r_arr / (p_arr + r_arr + 1e-9)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(thresholds_pr, p_arr,   label='Precision', color='steelblue')
ax.plot(thresholds_pr, r_arr,   label='Recall',    color='tomato')
ax.plot(thresholds_pr, f1_arr,  label='F1',        color='seagreen', linestyle='--')
ax.axvline(best_threshold, color='black', linestyle=':', lw=1.5,
           label=f'Tuned threshold ({best_threshold:.2f})')
ax.set_xlabel('Decision threshold')
ax.set_ylabel('Score')
ax.set_title('Precision / Recall / F1 vs. Decision Threshold')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.legend()
plt.tight_layout();

## 8. Feature importances (if supported)

In [ ]:
estimator = model.named_steps['model']
if hasattr(estimator, 'feature_importances_'):
    feature_names = model.named_steps['preprocess'].get_feature_names_out()
    top = (
        pd.DataFrame(
            {'feature': feature_names, 'importance': estimator.feature_importances_}
        )
        .sort_values('importance', ascending=False)
        .head(20)
    )
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.barplot(data=top, x='importance', y='feature', ax=ax, color='steelblue')
    ax.set_title('Top 20 feature importances')
    plt.tight_layout()
else:
    print('Best model does not expose feature_importances_.')

## 9. Interpretation

**Class imbalance** is the defining challenge: ~80% of BTS flights arrive on time,
so a naive model predicts on-time for everything and achieves high accuracy with
near-zero recall.  We address this three ways:
- `class_weight='balanced'` (LR, RF) / `scale_pos_weight` (XGBoost) up-weight
  the minority class during training.
- Threshold is tuned on the PR curve to target recall >= 40% (see section 7).
- ROC AUC is the selection metric because it is threshold-independent.

**Weather features** (`precip_in`, `visibility_mi`, `wind_speed_mph`) from the
NOAA CDO API are joined at the origin airport / departure hour level and improve
predictions on the days that matter most (storms, low visibility events).

**Departure hour, carrier, and origin airport** are usually the strongest
tree-based feature importances — flights leaving after 17:00 accumulate cascading
delays from earlier legs.

**Production-grade improvements** would include aircraft tail-number delay history,
time-based cross-validation (train on 2019–2022, test on 2023), and probability
calibration (Platt scaling / isotonic regression).